# BigMart Sales – v9 · Entity Embedding Neural Network
## 4th Base Learner + Learned Item/Outlet Embeddings → GBM Stack
**Hackathon: Data Science — Learn & Compete | April 2026**

---

### What's new in v9 vs v8 (RMSE 1174.49)

1. **`EntityEmbeddingNN`** — A PyTorch MLP with learned embedding tables for
   `Item_Identifier` (1,559 unique), `Outlet_Identifier` (10), `Item_Type` (16),
   and `Item_Category` (3). Trained with the same Grocery split, log1p target,
   and 5-fold OOF scheme as the GBM base learners.
2. **Embedding features** — The item and outlet embedding vectors are extracted
   from the trained NN and injected into the XGB/LGB/CatBoost feature sets.
   Learned embeddings capture latent structure (items that behave similarly across
   outlet types cluster together) that KFold TE cannot.
3. **MLP meta-learner** — A small 2-layer MLP replaces BayesianRidge. With 4 base
   OOF predictions + 2 raw features, it can learn non-linear combinations.
4. **4-model stacking** — OOF from XGB, LGB, CatBoost, and the NN feed the
   meta-learner. Lower inter-model correlation → better ensembling.

### Cell execution order (dependency map)
```
[1 Imports] → [2 DataLoader] → [3 Preprocessor] → [4 FeatureEngineer]
    → [5 Build CAT_COLS_NN encoders]   ← Must come BEFORE TE (raw cat cols still exist)
    → [6 KFold TE + CONT_COLS_NN]     ← Must come AFTER FE (TE_ cols needed)
    → [7 Grocery Split]
    → [8 NN Classes] → [9 Train NNs] → [10 Extract Embeddings]
    → [11 Optuna] → [12 Run Optuna]
    → [13 Stacking Classes] → [14 Fit Ensembles] → [15 Predict]
    → [16 Pseudo-labels] → [17 Visualisations] → [18 Summary]
```


## 0 · Install Dependencies

In [1]:
# Uncomment if running in a fresh environment
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu --quiet
# !pip install xgboost lightgbm catboost optuna scikit-learn pandas numpy --quiet


## 1 · Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader as TorchDataLoader, TensorDataset

import xgboost  as xgb
import lightgbm as lgb
import catboost as cbt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.base          import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics       import mean_squared_error

TRAIN_PATH = 'data/train_v9rqX0R.csv'
TEST_PATH  = 'data/test_AbJTz2l.csv'
SUB_PATH   = 'data/sample_submission_8RXa3c6.csv'
OUT_PATH   = 'outputs/submission_bigmart_v9_1.csv'
OUT_PATH2  = 'outputs/submission_bigmart_v9_pseudo.csv'

N_FOLDS          = 5
SEED             = 42
N_TRIALS_MAIN    = 100
N_TRIALS_GROCERY = 100

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs('outputs', exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'PyTorch  {torch.__version__}')
print(f'XGBoost  {xgb.__version__}')
print(f'LightGBM {lgb.__version__}')
print(f'CatBoost {cbt.__version__}')
print(f'Device   {DEVICE}')


## 2 · DataLoader

In [ ]:
class DataLoader:
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path  = test_path

    def load(self):
        train = pd.read_csv(self.train_path)
        test  = pd.read_csv(self.test_path)
        train['_split'] = 'train'
        test['_split']  = 'test'
        print(f'[DataLoader] train={train.shape}  test={test.shape}')
        return train, test

loader = DataLoader(TRAIN_PATH, TEST_PATH)
train_raw, test_raw = loader.load()
TARGET = 'Item_Outlet_Sales'


## 3 · Preprocessor

In [ ]:
class Preprocessor(BaseEstimator, TransformerMixin):
    _FAT_MAP = {'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'}

    def __init__(self):
        self._item_weight_map = {}; self._global_weight = 0.0
        self._item_vis_map    = {}; self._outlet_size_map = {}

    def fit(self, df, y=None):
        tmp = df.copy()
        tmp['Item_Fat_Content'] = tmp['Item_Fat_Content'].replace(self._FAT_MAP)
        self._item_weight_map = tmp.groupby('Item_Identifier')['Item_Weight'].mean().to_dict()
        self._global_weight   = tmp['Item_Weight'].mean()
        vis = tmp.copy()
        vis.loc[vis['Item_Visibility'] == 0, 'Item_Visibility'] = np.nan
        self._item_vis_map    = vis.groupby('Item_Identifier')['Item_Visibility'].mean().to_dict()
        self._outlet_size_map = (
            tmp.dropna(subset=['Outlet_Size'])
               .groupby('Outlet_Type')['Outlet_Size']
               .agg(lambda x: x.mode().iloc[0]).to_dict())
        return self

    def transform(self, df):
        df = df.copy()
        df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(self._FAT_MAP)
        df['Item_Weight'] = df.apply(
            lambda r: self._item_weight_map.get(r['Item_Identifier'], self._global_weight)
                      if pd.isna(r['Item_Weight']) else r['Item_Weight'], axis=1)
        df['Item_Visibility'] = df.apply(
            lambda r: self._item_vis_map.get(r['Item_Identifier'], r['Item_Visibility'])
                      if r['Item_Visibility'] == 0 else r['Item_Visibility'], axis=1)
        df['Outlet_Size_Was_Missing'] = df['Outlet_Size'].isna().astype(int)
        df['Outlet_Size'] = df.apply(
            lambda r: self._outlet_size_map.get(r['Outlet_Type'], 'Medium')
                      if pd.isna(r['Outlet_Size']) else r['Outlet_Size'], axis=1)
        df.loc[df['Item_Identifier'].str[:2] == 'NC', 'Item_Fat_Content'] = 'Non-Edible'
        return df

combined = pd.concat([train_raw, test_raw], ignore_index=True)
pre = Preprocessor()
pre.fit(combined)
combined_clean = pre.transform(combined)

train_clean = combined_clean[combined_clean['_split'] == 'train'].reset_index(drop=True)
test_clean  = combined_clean[combined_clean['_split'] == 'test'].reset_index(drop=True)

# Derive Item_Category here on train_clean / test_clean so it's available
# for both TE and the NN categorical encoders.
# (Item_Category is not in the raw data — it's derived from Item_Identifier prefix.)
def _derive_item_category(df):
    return df['Item_Identifier'].str[:2].map(
        {'FD': 'Food', 'DR': 'Drink', 'NC': 'Non-Consumable'}).fillna('Other')

train_clean['Item_Category'] = _derive_item_category(train_clean).values
test_clean['Item_Category']  = _derive_item_category(test_clean).values

print('Missing after preprocessing:')
print(train_clean[['Item_Weight', 'Item_Visibility', 'Outlet_Size']].isna().sum())
print()
print('Item_Category preview:', train_clean['Item_Category'].value_counts().to_dict())


## 4 · Feature Engineering

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    _ORDINAL_SIZE     = {'Small': 0, 'Medium': 1, 'High': 2}
    _ORDINAL_LOCATION = {'Tier 3': 0, 'Tier 2': 1, 'Tier 1': 2}
    _OUTLET_TYPE_ORD  = {'Grocery Store': 0, 'Supermarket Type1': 1,
                         'Supermarket Type2': 2, 'Supermarket Type3': 3}
    # Note: Item_Identifier, Outlet_Identifier kept in _DROP but intentionally
    # re-attached to train_fe after transform (needed by TE and NN encoders).
    _DROP = ['Item_Identifier', 'Outlet_Identifier',
             'Outlet_Establishment_Year', '_split']

    def __init__(self):
        self._label_encoders = {}
        self._item_vis_mean  = {}
        self._mrp_type_mean  = {}

    def _static(self, df):
        # Item_Category may already exist (added in Preprocessor cell); skip if so
        if 'Item_Category' not in df.columns:
            df['Item_Category'] = df['Item_Identifier'].str[:2].map(
                {'FD': 'Food', 'DR': 'Drink', 'NC': 'Non-Consumable'}).fillna('Other')
        df['Outlet_Age']         = 2013 - df['Outlet_Establishment_Year']
        df['Log_Item_Visibility']= np.log1p(df['Item_Visibility'])
        df['Outlet_Size_Ord']    = df['Outlet_Size'].map(self._ORDINAL_SIZE).fillna(1)
        df['Location_Type_Ord']  = df['Outlet_Location_Type'].map(self._ORDINAL_LOCATION).fillna(0)
        df['Outlet_Type_Ord']    = df['Outlet_Type'].map(self._OUTLET_TYPE_ORD).fillna(0)
        df['Is_Grocery']         = (df['Outlet_Type'] == 'Grocery Store').astype(int)
        df['Item_Cat_x_Outlet']  = df['Item_Category'] + '_' + df['Outlet_Type']
        df['MRP_Bin']            = pd.qcut(df['Item_MRP'], q=16, labels=False, duplicates='drop')
        df['Outlet_Age_x_Type']  = df['Outlet_Age'] * df['Outlet_Type_Ord']
        return df

    def fit(self, df, y=None):
        tmp = self._static(df.copy())
        self._item_vis_mean = df.groupby('Item_Identifier')['Item_Visibility'].mean().to_dict()
        self._mrp_type_mean = df.groupby('Item_Type')['Item_MRP'].mean().to_dict()
        obj_cols = [c for c in tmp.select_dtypes('object').columns
                    if c not in [TARGET, 'Item_Identifier', 'Outlet_Identifier', '_split']]
        for col in obj_cols:
            le = LabelEncoder()
            le.fit(tmp[col].astype(str))
            self._label_encoders[col] = le
        return self

    def transform(self, df):
        df = df.copy()
        df = self._static(df)
        df['Item_Visibility_MeanRatio'] = df.apply(
            lambda r: r['Item_Visibility'] /
                      self._item_vis_mean.get(r['Item_Identifier'], r['Item_Visibility'] + 1e-6),
            axis=1)
        df['MRP_Per_Category'] = df.apply(
            lambda r: r['Item_MRP'] /
                      self._mrp_type_mean.get(r['Item_Type'], r['Item_MRP'] + 1e-6),
            axis=1)
        df['MRP_x_OutletType'] = df['Item_MRP'] * df['Outlet_Type_Ord']
        df['Weight_x_MRP']     = df['Item_Weight'] * df['Item_MRP']
        for col, le in self._label_encoders.items():
            if col in df.columns:
                df[col] = le.transform(df[col].astype(str))
        df = df.drop(columns=[c for c in self._DROP if c in df.columns], errors='ignore')
        if TARGET in df.columns:
            df = df.drop(columns=[TARGET])
        return df

fe = FeatureEngineer()
fe.fit(pd.concat([train_clean, test_clean], ignore_index=True))
train_fe = fe.transform(train_clean)
test_fe  = fe.transform(test_clean)

print(f'Feature-engineered shape: {train_fe.shape}')
print('Sample columns:', list(train_fe.columns)[:8], '...')


## 5 · Build NN Categorical Encoders
**Must run before KFold TE** — TE will drop `Item_Identifier`, `Outlet_Identifier`,
`Item_Type`, and `Item_Category` from `train_fe`. We source these from `train_clean`
(which still has them), and re-attach them to `train_fe` below for the TE to encode.

`Item_Category` was derived in the Preprocessor cell and is already present on
`train_clean` / `test_clean`.


In [ ]:
# ── Integer encoders for NN embedding tables ─────────────────────────────────
# Source: train_clean / test_clean (pre-FE, so raw cat cols still present).
# Item_Category was derived from Item_Identifier in the Preprocessor cell.

CAT_COLS_NN = {
    'Item_Identifier'  : {'col': 'Item_Identifier',   'embed_dim': 20},
    'Outlet_Identifier': {'col': 'Outlet_Identifier',  'embed_dim': 5},
    'Item_Type'        : {'col': 'Item_Type',           'embed_dim': 6},
    'Item_Category'    : {'col': 'Item_Category',       'embed_dim': 3},
}

cat_encoders_nn = {}
for key, cfg in CAT_COLS_NN.items():
    col = cfg['col']
    le  = LabelEncoder()
    combined_vals = pd.concat([train_clean[col], test_clean[col]], ignore_index=True)
    le.fit(combined_vals.astype(str))
    cat_encoders_nn[col] = le
    cfg['n_categories']  = len(le.classes_)
    print(f'{col:25s}  n={cfg["n_categories"]:5d}  embed_dim={cfg["embed_dim"]}')

# Re-attach raw cat cols to train_fe / test_fe so that KFold TE can encode them.
# This is the same pattern used in v8.
for col in ['Item_Identifier', 'Outlet_Identifier', 'Item_Type',
            'Item_Category', 'Item_Cat_x_Outlet']:
    if col in train_clean.columns:
        train_fe[col] = train_clean[col].values
    if col in test_clean.columns:
        test_fe[col]  = test_clean[col].values

print('\nRaw cat cols re-attached to train_fe / test_fe for TE.')
print(f'train_fe shape after re-attach: {train_fe.shape}')


## 6 · KFold Target Encoding + CONT_COLS_NN

In [ ]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    ENCODE_COLS = ['Item_Identifier', 'Outlet_Identifier',
                   'Item_Type', 'Item_Category', 'Item_Cat_x_Outlet']

    def __init__(self, n_folds=5, alpha=10.0, seed=42):
        self.n_folds = n_folds; self.alpha = alpha; self.seed = seed
        self._global_mean = 0.0; self._encode_maps = {}

    def fit(self, train_df, target_series):
        self._global_mean = target_series.mean()
        for col in self.ENCODE_COLS:
            if col not in train_df.columns: continue
            stats = target_series.groupby(train_df[col]).agg(['sum', 'count'])
            smoothed = ((stats['sum'] + self.alpha * self._global_mean) /
                        (stats['count'] + self.alpha))
            self._encode_maps[col] = smoothed.to_dict()
        return self

    def transform_train(self, train_df, target_series):
        df = train_df.copy()
        kf = KFold(self.n_folds, shuffle=True, random_state=self.seed)
        for col in self.ENCODE_COLS:
            if col not in df.columns: continue
            encoded = np.zeros(len(df))
            for tr_idx, va_idx in kf.split(df):
                fold_stats = target_series.iloc[tr_idx].groupby(
                    df[col].iloc[tr_idx]).agg(['sum', 'count'])
                smoothed = ((fold_stats['sum'] + self.alpha * self._global_mean) /
                            (fold_stats['count'] + self.alpha))
                encoded[va_idx] = df[col].iloc[va_idx].map(smoothed).fillna(
                    self._global_mean).values
            df[f'TE_{col}'] = encoded
            df = df.drop(columns=[col])
        return df

    def transform_test(self, test_df):
        df = test_df.copy()
        for col in self.ENCODE_COLS:
            if col not in df.columns: continue
            df[f'TE_{col}'] = df[col].map(self._encode_maps[col]).fillna(self._global_mean)
            df = df.drop(columns=[col])
        return df

y_train_log = np.log1p(train_clean[TARGET].values)

te = KFoldTargetEncoder(n_folds=N_FOLDS, alpha=10.0, seed=SEED)
te.fit(train_fe, pd.Series(y_train_log))
train_final = te.transform_train(train_fe, pd.Series(y_train_log))
test_final  = te.transform_test(test_fe)

# Drop consensus-useless features from v8 permutation analysis
DROP_FEATURES = ['Is_Grocery', 'Item_Fat_Content']
train_final = train_final.drop(columns=[c for c in DROP_FEATURES if c in train_final.columns])
test_final  = test_final.drop(columns=[c for c in DROP_FEATURES if c in test_final.columns])

FEATURES = [c for c in train_final.columns if c != TARGET]
print(f'Final GBM feature count: {len(FEATURES)}')

# ── CONT_COLS_NN: defined HERE (after TE) so TE_ columns exist in train_final ─
CONT_COLS_NN = [
    'Item_MRP', 'Item_Weight', 'Item_Visibility', 'Log_Item_Visibility',
    'Item_Visibility_MeanRatio', 'MRP_Per_Category', 'MRP_x_OutletType',
    'Weight_x_MRP', 'Outlet_Age', 'Outlet_Age_x_Type',
    'Outlet_Size_Ord', 'Location_Type_Ord', 'Outlet_Type_Ord',
    'Outlet_Size_Was_Missing', 'MRP_Bin',
    'TE_Item_Identifier', 'TE_Outlet_Identifier',
    'TE_Item_Type', 'TE_Item_Category', 'TE_Item_Cat_x_Outlet',
]
CONT_COLS_NN = [c for c in CONT_COLS_NN if c in train_final.columns]
print(f'Continuous features for NN: {len(CONT_COLS_NN)}')

missing_te = [c for c in ['TE_Item_Identifier', 'TE_Outlet_Identifier'] if c not in CONT_COLS_NN]
if missing_te:
    print(f'WARNING: missing TE cols in CONT_COLS_NN: {missing_te}')
else:
    print('All expected TE cols present in CONT_COLS_NN ✓')


## 7 · Grocery / Supermarket Split

In [ ]:
# Store grocery mask on train/test before dropping the Is_Grocery column
train_final['_Is_Grocery'] = (train_clean['Outlet_Type'] == 'Grocery Store').astype(int).values
test_final['_Is_Grocery']  = (test_clean['Outlet_Type']  == 'Grocery Store').astype(int).values

grocery_mask_train = train_final['_Is_Grocery'] == 1
grocery_mask_test  = test_final['_Is_Grocery']  == 1

train_final = train_final.drop(columns=['_Is_Grocery'])
test_final  = test_final.drop(columns=['_Is_Grocery'])

# Per-split dataframes — both the GBM feature matrix and the raw clean df
# (the raw clean df is needed by build_nn_tensors for categorical encoding)
train_clean_sup  = train_clean[~grocery_mask_train].reset_index(drop=True)
train_clean_groc = train_clean[grocery_mask_train].reset_index(drop=True)
train_final_sup  = train_final[FEATURES][~grocery_mask_train].reset_index(drop=True)
train_final_groc = train_final[FEATURES][grocery_mask_train].reset_index(drop=True)

test_clean_sup   = test_clean[~grocery_mask_test].reset_index(drop=True)
test_clean_groc  = test_clean[grocery_mask_test].reset_index(drop=True)
test_final_sup   = test_final[FEATURES][~grocery_mask_test].reset_index(drop=True)
test_final_groc  = test_final[FEATURES][grocery_mask_test].reset_index(drop=True)

X_sup  = train_final_sup.values;  y_sup  = y_train_log[~grocery_mask_train]
X_groc = train_final_groc.values; y_groc = y_train_log[grocery_mask_train]

X_test_sup  = test_final_sup.values
X_test_groc = test_final_groc.values

print(f'Supermarket train: {len(y_sup):,} rows  mean sales={np.expm1(y_sup).mean():.0f}')
print(f'Grocery     train: {len(y_groc):,} rows  mean sales={np.expm1(y_groc).mean():.0f}')


## 8 · Entity Embedding Neural Network (PyTorch)

Architecture per split:
```
[Embeddings: item(20) + outlet(5) + type(6) + category(3)]
  + [ContBN: 20 numeric features]
  → FC(512) → BN → ReLU → Dropout(0.3)
  → FC(256) → BN → ReLU → Dropout(0.3)
  → FC(128) → BN → ReLU → Dropout(0.3)
  → FC(1)  [predicts log1p(sales)]
```
Embedding dim rule: `min(50, n_categories // 2 + 1)` — already applied above.


In [ ]:
def build_nn_tensors(df_final, df_clean, cat_encoders, cat_cols_cfg, cont_cols, y=None):
    """
    Encodes categorical columns via pre-fitted LabelEncoders (+1 offset for
    padding_idx=0), stacks continuous features, and returns PyTorch tensors.

    df_final  : DataFrame with CONT_COLS_NN columns (post-TE GBM features)
    df_clean  : DataFrame with raw categorical columns (train_clean / test_clean)
    """
    cat_arrays = []
    for col in cat_cols_cfg.keys():
        enc    = cat_encoders[col]
        vals   = df_clean[col].astype(str).values
        # +1 so that index 0 is reserved for padding_idx in nn.Embedding
        encoded = enc.transform(vals) + 1
        cat_arrays.append(encoded)

    X_cat  = torch.tensor(np.stack(cat_arrays, axis=1), dtype=torch.long)
    X_cont = torch.tensor(df_final[cont_cols].values.astype(np.float32),
                          dtype=torch.float32)
    if y is not None:
        Y = torch.tensor(y.astype(np.float32), dtype=torch.float32)
        return X_cat, X_cont, Y
    return X_cat, X_cont


class EntityEmbeddingNN(nn.Module):
    def __init__(self, cat_configs, n_cont, hidden_dims=(512, 256, 128), dropout=0.3):
        super().__init__()
        self.embeddings     = nn.ModuleDict()
        self._cat_col_order = list(cat_configs.keys())
        embed_total_dim     = 0
        for col, cfg in cat_configs.items():
            self.embeddings[col] = nn.Embedding(
                cfg['n_categories'] + 1, cfg['embed_dim'], padding_idx=0)
            embed_total_dim += cfg['embed_dim']

        self.cont_bn = nn.BatchNorm1d(n_cont)
        in_dim       = embed_total_dim + n_cont
        layers = []
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.fc = nn.Sequential(*layers)

    def forward(self, x_cat, x_cont):
        embeds = [self.embeddings[col](x_cat[:, i])
                  for i, col in enumerate(self._cat_col_order)]
        x = torch.cat(embeds + [self.cont_bn(x_cont)], dim=1)
        return self.fc(x).squeeze(1)


class NNTrainer:
    """
    5-fold OOF training for EntityEmbeddingNN.
    Stores all 5 fold models for test-time averaging and embedding extraction.
    """
    def __init__(self, cat_configs, cont_cols, cat_encoders,
                 epochs=80, batch_size=256, lr=1e-3, dropout=0.3,
                 n_folds=5, seed=42, device=DEVICE):
        self.cat_configs  = cat_configs;  self.cont_cols = cont_cols
        self.cat_encoders = cat_encoders; self.epochs    = epochs
        self.batch_size   = batch_size;   self.lr        = lr
        self.dropout      = dropout;      self.n_folds   = n_folds
        self.seed         = seed;         self.device    = device
        self.fold_models  = []

    def _train_one(self, model, X_cat_tr, X_cont_tr, Y_tr,
                   X_cat_va, X_cont_va, Y_va):
        model     = model.to(self.device)
        optimizer = optim.AdamW(model.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)
        criterion = nn.MSELoss()
        ds_tr     = TensorDataset(X_cat_tr.to(self.device),
                                  X_cont_tr.to(self.device),
                                  Y_tr.to(self.device))
        loader    = TorchDataLoader(ds_tr, batch_size=self.batch_size, shuffle=True)
        best_val, best_state, patience = np.inf, None, 0
        PATIENCE  = 15

        for epoch in range(self.epochs):
            model.train()
            for xc, xn, yb in loader:
                optimizer.zero_grad()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                criterion(model(xc, xn), yb).backward()
                optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                val_pred = model(X_cat_va.to(self.device),
                                 X_cont_va.to(self.device)).cpu().numpy()
            val_rmse = np.sqrt(mean_squared_error(Y_va.numpy(), val_pred))
            if val_rmse < best_val:
                best_val  = val_rmse
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience  = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        model.load_state_dict(best_state)
        return model, best_val

    def fit_cv(self, df_final, df_clean, y, split_label=''):
        X_cat, X_cont, Y = build_nn_tensors(
            df_final, df_clean, self.cat_encoders,
            self.cat_configs, self.cont_cols, y)

        kf = KFold(self.n_folds, shuffle=True, random_state=self.seed)
        oof = np.zeros(len(y))
        self.fold_models = []

        for fold, (tr_idx, va_idx) in enumerate(kf.split(X_cat), 1):
            model = EntityEmbeddingNN(
                self.cat_configs, len(self.cont_cols), dropout=self.dropout)
            model, val_rmse = self._train_one(
                model,
                X_cat[tr_idx], X_cont[tr_idx], Y[tr_idx],
                X_cat[va_idx], X_cont[va_idx], Y[va_idx])
            model.eval()
            with torch.no_grad():
                oof[va_idx] = model(
                    X_cat[va_idx].to(self.device),
                    X_cont[va_idx].to(self.device)).cpu().numpy()
            self.fold_models.append(model)
            print(f'  [{split_label}] Fold {fold}/5  best val RMSE(log)={val_rmse:.5f}')

        oof_rmse = np.sqrt(mean_squared_error(y, oof))
        raw_rmse = np.sqrt(mean_squared_error(np.expm1(y), np.expm1(oof)))
        print(f'  [{split_label}] OOF RMSE(log)={oof_rmse:.5f}  raw={raw_rmse:.2f}')
        return oof, oof_rmse

    def predict(self, df_final, df_clean):
        X_cat, X_cont = build_nn_tensors(
            df_final, df_clean, self.cat_encoders,
            self.cat_configs, self.cont_cols)
        preds = []
        for m in self.fold_models:
            m.eval()
            with torch.no_grad():
                preds.append(m(X_cat.to(self.device),
                               X_cont.to(self.device)).cpu().numpy())
        return np.mean(preds, axis=0)

    def get_embeddings(self, df_clean, col):
        """Return averaged embedding weight matrix across all fold models."""
        return np.mean(
            [m.embeddings[col].weight.detach().cpu().numpy()
             for m in self.fold_models], axis=0)


print('EntityEmbeddingNN + NNTrainer defined.')


## 9 · Train NN on Both Splits

In [ ]:
print('=' * 60)
print('SUPERMARKET — Training Entity Embedding NN')
print('=' * 60)
nn_trainer_sup = NNTrainer(
    CAT_COLS_NN, CONT_COLS_NN, cat_encoders_nn,
    epochs=100, batch_size=256, lr=1e-3, dropout=0.3,
    n_folds=N_FOLDS, seed=SEED, device=DEVICE)
oof_nn_sup, rmse_nn_sup = nn_trainer_sup.fit_cv(
    train_final_sup, train_clean_sup, y_sup, split_label='SUP')

print()
print('=' * 60)
print('GROCERY — Training Entity Embedding NN')
print('=' * 60)
nn_trainer_groc = NNTrainer(
    CAT_COLS_NN, CONT_COLS_NN, cat_encoders_nn,
    epochs=100, batch_size=64, lr=5e-4, dropout=0.2,
    n_folds=N_FOLDS, seed=SEED, device=DEVICE)
oof_nn_groc, rmse_nn_groc = nn_trainer_groc.fit_cv(
    train_final_groc, train_clean_groc, y_groc, split_label='GROC')


## 10 · Extract Learned Embeddings as Features

After NN training, the embedding weight matrices are averaged across all 5
fold models and each row is mapped to its learned vector.  These dense features
are appended to the GBM feature matrix so XGB/LGB/CatBoost can exploit the
latent structure the NN discovered.


In [ ]:
def add_embedding_features(df_final, df_clean, trainer):
    """
    Appends item/outlet/type/category embedding vectors as new columns.

    df_final : per-split GBM feature DataFrame (e.g. train_final_sup)
    df_clean : per-split raw-clean DataFrame   (e.g. train_clean_sup)
               Must contain the categorical columns in CAT_COLS_NN.
    """
    df = df_final.copy()
    for col, cfg in CAT_COLS_NN.items():
        emb_matrix = trainer.get_embeddings(df_clean, col)  # (n_cats+1, embed_dim)
        enc        = cat_encoders_nn[col]
        indices    = enc.transform(df_clean[col].astype(str).values) + 1  # +1 for padding offset
        emb_feats  = emb_matrix[indices]                                   # (n_rows, embed_dim)
        for d in range(emb_feats.shape[1]):
            df[f'EMB_{col}_{d}'] = emb_feats[:, d]
    return df

# Add embedding features to all four subsets
train_final_sup_emb  = add_embedding_features(train_final_sup,  train_clean_sup,  nn_trainer_sup)
train_final_groc_emb = add_embedding_features(train_final_groc, train_clean_groc, nn_trainer_groc)
test_final_sup_emb   = add_embedding_features(test_final_sup,   test_clean_sup,   nn_trainer_sup)
test_final_groc_emb  = add_embedding_features(test_final_groc,  test_clean_groc,  nn_trainer_groc)

FEATURES_EMB  = [c for c in train_final_sup_emb.columns if c != TARGET]
n_emb_added   = len(FEATURES_EMB) - len(FEATURES)

print(f'Original GBM features     : {len(FEATURES)}')
print(f'Embedding features added  : {n_emb_added}')
print(f'Total features for GBMs   : {len(FEATURES_EMB)}')

# Sanity: no NaNs introduced
for name, df in [('sup_train', train_final_sup_emb),
                 ('groc_train', train_final_groc_emb),
                 ('sup_test',   test_final_sup_emb),
                 ('groc_test',  test_final_groc_emb)]:
    nans = df[FEATURES_EMB].isna().sum().sum()
    print(f'  {name}: NaNs={nans}')
    assert nans == 0, f'NaNs in {name}!'


## 11 · Optuna Hyperparameter Tuning (GBMs)

In [ ]:
class OptunaTuner:
    def __init__(self, model_type, n_trials=100, n_folds=5, seed=42):
        assert model_type in ('xgb', 'lgb', 'cat')
        self.model_type = model_type; self.n_trials = n_trials
        self.n_folds    = n_folds;    self.seed     = seed
        self.study      = None;       self.best_params_ = {}

    def _suggest_xgb(self, trial):
        return dict(
            n_estimators     =trial.suggest_int  ('n_estimators',    300,2000),
            max_depth        =trial.suggest_int  ('max_depth',         3,  10),
            learning_rate    =trial.suggest_float('learning_rate',  .005, .3, log=True),
            subsample        =trial.suggest_float('subsample',        .5, 1.),
            colsample_bytree =trial.suggest_float('colsample_bytree', .5, 1.),
            colsample_bylevel=trial.suggest_float('colsample_bylevel',.5, 1.),
            min_child_weight =trial.suggest_int  ('min_child_weight',   1, 20),
            gamma            =trial.suggest_float('gamma',            .0, 1.),
            reg_alpha        =trial.suggest_float('reg_alpha',      1e-8,10., log=True),
            reg_lambda       =trial.suggest_float('reg_lambda',     1e-8,10., log=True),
            tree_method='hist', random_state=self.seed, n_jobs=-1, verbosity=0)

    def _suggest_lgb(self, trial):
        return dict(
            n_estimators     =trial.suggest_int  ('n_estimators',    300,2000),
            max_depth        =trial.suggest_int  ('max_depth',         3,  12),
            learning_rate    =trial.suggest_float('learning_rate',  .005, .3, log=True),
            num_leaves       =trial.suggest_int  ('num_leaves',       20, 300),
            subsample        =trial.suggest_float('subsample',        .5, 1.),
            colsample_bytree =trial.suggest_float('colsample_bytree', .5, 1.),
            min_child_samples=trial.suggest_int  ('min_child_samples',  5,100),
            reg_alpha        =trial.suggest_float('reg_alpha',      1e-8,10., log=True),
            reg_lambda       =trial.suggest_float('reg_lambda',     1e-8,10., log=True),
            random_state=self.seed, n_jobs=-1, verbosity=-1)

    def _suggest_cat(self, trial):
        return dict(
            iterations          =trial.suggest_int  ('iterations',     300,2000),
            depth               =trial.suggest_int  ('depth',            3,  10),
            learning_rate       =trial.suggest_float('learning_rate', .005, .3, log=True),
            l2_leaf_reg         =trial.suggest_float('l2_leaf_reg',  1e-8,10., log=True),
            bagging_temperature =trial.suggest_float('bagging_temperature', .0, 1.),
            random_strength     =trial.suggest_float('random_strength',     .5, 5.),
            border_count        =trial.suggest_int  ('border_count',    32, 255),
            random_seed=self.seed, verbose=0)

    def _make(self, params):
        if self.model_type == 'xgb': return xgb.XGBRegressor(**params)
        if self.model_type == 'lgb': return lgb.LGBMRegressor(**params)
        return cbt.CatBoostRegressor(**params)

    def _objective(self, trial, X, y):
        params = {'xgb':self._suggest_xgb,'lgb':self._suggest_lgb,
                  'cat':self._suggest_cat}[self.model_type](trial)
        kf = KFold(self.n_folds, shuffle=True, random_state=self.seed)
        rmses = []
        for tr, va in kf.split(X):
            m = self._make(params)
            m.fit(X[tr], y[tr])
            rmses.append(np.sqrt(mean_squared_error(y[va], m.predict(X[va]))))
        return np.mean(rmses)

    def tune(self, X, y, study_name=''):
        self.study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=self.seed),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
            study_name=study_name)
        self.study.optimize(lambda t: self._objective(t, X, y),
                            n_trials=self.n_trials, show_progress_bar=True)
        self.best_params_ = self.study.best_params
        print(f'  Best OOF RMSE(log): {self.study.best_value:.5f}')
        return self.best_params_

print('OptunaTuner defined.')


## 12 · Run Optuna (GBMs on embedding-augmented features)

In [ ]:
X_sup_emb  = train_final_sup_emb[FEATURES_EMB].values
X_groc_emb = train_final_groc_emb[FEATURES_EMB].values

print('=' * 60)
print('SUPERMARKET — tuning XGB, LGB, CatBoost')
print('=' * 60)
tuner_xgb_sup  = OptunaTuner('xgb', n_trials=N_TRIALS_MAIN,    seed=SEED)
params_xgb_sup = tuner_xgb_sup.tune(X_sup_emb, y_sup, 'xgb_sup_v9')
tuner_lgb_sup  = OptunaTuner('lgb', n_trials=N_TRIALS_MAIN,    seed=SEED)
params_lgb_sup = tuner_lgb_sup.tune(X_sup_emb, y_sup, 'lgb_sup_v9')
tuner_cat_sup  = OptunaTuner('cat', n_trials=N_TRIALS_MAIN,    seed=SEED)
params_cat_sup = tuner_cat_sup.tune(X_sup_emb, y_sup, 'cat_sup_v9')

print()
print('=' * 60)
print('GROCERY — tuning XGB, LGB, CatBoost')
print('=' * 60)
tuner_xgb_groc  = OptunaTuner('xgb', n_trials=N_TRIALS_GROCERY, seed=SEED)
params_xgb_groc = tuner_xgb_groc.tune(X_groc_emb, y_groc, 'xgb_groc_v9')
tuner_lgb_groc  = OptunaTuner('lgb', n_trials=N_TRIALS_GROCERY, seed=SEED)
params_lgb_groc = tuner_lgb_groc.tune(X_groc_emb, y_groc, 'lgb_groc_v9')
tuner_cat_groc  = OptunaTuner('cat', n_trials=N_TRIALS_GROCERY, seed=SEED)
params_cat_groc = tuner_cat_groc.tune(X_groc_emb, y_groc, 'cat_groc_v9')


## 13 · 4-Model Stacking (XGB + LGB + CatBoost + NN)

Meta-learner is a small MLP (6 → 32 → 32 → 1):
- **Inputs**: 4 OOF predictions + `Item_MRP` + `TE_Outlet_Identifier`
- All 6 inputs are passed through `StandardScaler` before the MLP


In [ ]:
class MLPMeta(nn.Module):
    """Tiny MLP meta-learner. 6 → 32 → 32 → 1."""
    def __init__(self, n_inputs=6, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_inputs, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden,   hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden,   1))

    def forward(self, x):
        return self.net(x).squeeze(1)


class StackingEnsemble4:
    """
    4-base stacking: XGB, LGB, CatBoost, EntityEmbeddingNN.
    Meta-learner: small MLP trained on [4 OOF preds + Item_MRP + TE_Outlet_Identifier].

    Meta OOF RMSE is estimated via a held-out fold (1/5 of the OOF stack),
    so the reported RMSE is never in-sample.
    """
    def __init__(self, params_xgb, params_lgb, params_cat,
                 nn_trainer, nn_df_final, nn_df_clean,
                 features_emb, cont_cols_nn,
                 n_folds=5, seed=42):
        self.params       = {'xgb': params_xgb, 'lgb': params_lgb, 'cat': params_cat}
        self.nn_trainer   = nn_trainer
        self.nn_df_final  = nn_df_final
        self.nn_df_clean  = nn_df_clean
        self.features_emb = features_emb
        self.n_folds      = n_folds;  self.seed = seed
        self._mrp_idx     = features_emb.index('Item_MRP')
        self._te_out_idx  = features_emb.index('TE_Outlet_Identifier')
        self._base_models_final = {}
        self._meta_scaler = StandardScaler()
        self._meta_model  = None

    def _make_gbm(self, mtype, params):
        if mtype == 'xgb': return xgb.XGBRegressor(**params)
        if mtype == 'lgb': return lgb.LGBMRegressor(**params)
        return cbt.CatBoostRegressor(**params)

    def _train_meta_model(self, Zs, y):
        """Train MLP meta on scaled inputs. Returns fitted model."""
        model     = MLPMeta(n_inputs=Zs.shape[1]).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        ds        = TensorDataset(torch.tensor(Zs, dtype=torch.float32),
                                  torch.tensor(y,  dtype=torch.float32))
        loader    = TorchDataLoader(ds, batch_size=128, shuffle=True)
        for _ in range(200):
            model.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
                optimizer.step()
        return model

    def fit_cv(self, X, y):
        kf  = KFold(self.n_folds, shuffle=True, random_state=self.seed)
        oof = np.zeros((len(y), 4))   # base model OOF predictions

        # ── Step 1: generate base model OOF predictions ───────────────────────
        for fold, (tr, va) in enumerate(kf.split(X), 1):
            for i, mtype in enumerate(['xgb', 'lgb', 'cat']):
                m = self._make_gbm(mtype, self.params[mtype])
                m.fit(X[tr], y[tr])
                oof[va, i] = m.predict(X[va])
            va_df_final = self.nn_df_final.iloc[va].reset_index(drop=True)
            va_df_clean = self.nn_df_clean.iloc[va].reset_index(drop=True)
            oof[va, 3]  = self.nn_trainer.predict(va_df_final, va_df_clean)
            print(f'  Fold {fold}/5', end='\r')
        print()

        # Diagnostic: OOF pairwise correlations
        corr   = np.corrcoef(oof.T)
        labels = ['XGB', 'LGB', 'CAT', 'NN']
        print('OOF pairwise correlations:')
        for i in range(4):
            for j in range(i + 1, 4):
                print(f'  {labels[i]}-{labels[j]}: {corr[i,j]:.4f}')

        # Base model simple average OOF RMSE (honest, no meta overfitting)
        oof_avg      = oof.mean(axis=1)
        base_raw_rmse = np.sqrt(mean_squared_error(np.expm1(y), np.expm1(oof_avg)))
        print(f'Base avg OOF RMSE (raw): {base_raw_rmse:.2f}')

        # ── Step 2: train meta + estimate meta RMSE with held-out fold ────────
        # Build full meta input matrix
        Z = np.column_stack([oof, X[:, self._mrp_idx], X[:, self._te_out_idx]])

        # Held-out meta RMSE: train meta on 4 folds, evaluate on 1 fold
        meta_oof = np.zeros(len(y))
        meta_scaler_cv = StandardScaler()
        for tr, va in kf.split(Z):
            Zs_tr = meta_scaler_cv.fit_transform(Z[tr])
            Zs_va = meta_scaler_cv.transform(Z[va])
            m_tmp = self._train_meta_model(Zs_tr, y[tr])
            m_tmp.eval()
            with torch.no_grad():
                meta_oof[va] = m_tmp(
                    torch.tensor(Zs_va, dtype=torch.float32).to(DEVICE)).cpu().numpy()

        meta_raw_rmse = np.sqrt(mean_squared_error(np.expm1(y), np.expm1(meta_oof)))
        print(f'Meta  OOF RMSE (raw):   {meta_raw_rmse:.2f}')

        # ── Step 3: train final meta on ALL OOF data ──────────────────────────
        Zs_full          = self._meta_scaler.fit_transform(Z)
        self._meta_model = self._train_meta_model(Zs_full, y)

        return meta_raw_rmse, meta_oof

    def fit_full(self, X, y):
        """Refit all base GBMs on full data."""
        for mtype in ['xgb', 'lgb', 'cat']:
            m = self._make_gbm(mtype, self.params[mtype])
            m.fit(X, y)
            self._base_models_final[mtype] = m
        print('Base GBMs refit on full data.')

    def predict(self, X, nn_df_final_test, nn_df_clean_test):
        gbm_preds = np.stack(
            [self._base_models_final[m].predict(X) for m in ['xgb', 'lgb', 'cat']], axis=1)
        nn_pred   = self.nn_trainer.predict(nn_df_final_test, nn_df_clean_test)
        Z  = np.column_stack([gbm_preds, nn_pred,
                              X[:, self._mrp_idx], X[:, self._te_out_idx]])
        Zs = self._meta_scaler.transform(Z)
        self._meta_model.eval()
        with torch.no_grad():
            preds = self._meta_model(
                torch.tensor(Zs, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        return np.expm1(preds)

print('MLPMeta + StackingEnsemble4 defined.')


## 14 · Fit & Evaluate Stacking Ensembles

In [ ]:
print('=' * 60)
print('SUPERMARKET — 4-model stacking CV')
print('=' * 60)
ens_sup = StackingEnsemble4(
    params_xgb_sup, params_lgb_sup, params_cat_sup,
    nn_trainer   = nn_trainer_sup,
    nn_df_final  = train_final_sup_emb[FEATURES_EMB],
    nn_df_clean  = train_clean_sup,
    features_emb = FEATURES_EMB,
    cont_cols_nn = CONT_COLS_NN,
    n_folds=N_FOLDS, seed=SEED)
rmse_sup, oof_sup = ens_sup.fit_cv(X_sup_emb, y_sup)

print()
print('=' * 60)
print('GROCERY — 4-model stacking CV')
print('=' * 60)
ens_groc = StackingEnsemble4(
    params_xgb_groc, params_lgb_groc, params_cat_groc,
    nn_trainer   = nn_trainer_groc,
    nn_df_final  = train_final_groc_emb[FEATURES_EMB],
    nn_df_clean  = train_clean_groc,
    features_emb = FEATURES_EMB,
    cont_cols_nn = CONT_COLS_NN,
    n_folds=N_FOLDS, seed=SEED)
rmse_groc, oof_groc = ens_groc.fit_cv(X_groc_emb, y_groc)

n_sup  = len(y_sup);  n_groc = len(y_groc)
combined_rmse = (rmse_sup * n_sup + rmse_groc * n_groc) / (n_sup + n_groc)

print()
print(f'Supermarket OOF RMSE: {rmse_sup:,.2f}  (n={n_sup:,})')
print(f'Grocery     OOF RMSE: {rmse_groc:,.2f}  (n={n_groc:,})')
print(f'Combined    OOF RMSE: {combined_rmse:,.2f}')
print(f'v8 baseline         : 1174.49')
print(f'Delta vs v8         : {combined_rmse - 1174.49:+.2f}')


## 15 · Refit on Full Data & Generate Predictions

In [ ]:
print('Refitting Supermarket ensemble on full data ...')
ens_sup.fit_full(X_sup_emb, y_sup)

print('Refitting Grocery ensemble on full data ...')
ens_groc.fit_full(X_groc_emb, y_groc)

preds_sup  = ens_sup.predict(
    test_final_sup_emb[FEATURES_EMB].values,
    test_final_sup_emb[FEATURES_EMB],
    test_clean_sup)

preds_groc = ens_groc.predict(
    test_final_groc_emb[FEATURES_EMB].values,
    test_final_groc_emb[FEATURES_EMB],
    test_clean_groc)

# Reconstruct full-test prediction array
sub          = pd.read_csv(SUB_PATH)
final_preds  = np.zeros(len(test_final))
sup_idx      = test_final.index[~grocery_mask_test].tolist()
groc_idx     = test_final.index[grocery_mask_test].tolist()
final_preds[sup_idx]  = preds_sup
final_preds[groc_idx] = preds_groc

MIN_SALES   = train_raw[TARGET].min()
final_preds = np.clip(final_preds, MIN_SALES, None)
print(f'Predictions clipped at {MIN_SALES:.2f}')

sub[TARGET] = final_preds
sub.to_csv(OUT_PATH, index=False)
print(f'Round-1 submission saved → {OUT_PATH}')


## 16 · Pseudo-labelling Round

In [ ]:
print('Running pseudo-label round ...')
mean_pred       = final_preds.mean()
std_pred        = final_preds.std()
confident_mask  = np.abs(final_preds - mean_pred) > 0.5 * std_pred
print(f'Confident test rows: {confident_mask.sum()} / {len(final_preds)}')

test_pseudo          = test_clean.copy()
test_pseudo[TARGET]  = final_preds
test_pseudo_conf     = test_pseudo[confident_mask].reset_index(drop=True)
train_aug            = pd.concat([train_clean, test_pseudo_conf], ignore_index=True)
y_aug_log            = np.log1p(train_aug[TARGET].values)

# Rebuild FE + TE on augmented data
combined_aug_clean   = pre.transform(pd.concat([train_aug, test_clean], ignore_index=True))
train_aug_clean      = combined_aug_clean.iloc[:len(train_aug)].reset_index(drop=True)
test_aug_clean       = combined_aug_clean.iloc[len(train_aug):].reset_index(drop=True)
train_aug_clean['Item_Category'] = _derive_item_category(train_aug_clean).values
test_aug_clean['Item_Category']  = _derive_item_category(test_aug_clean).values

fe2 = FeatureEngineer()
fe2.fit(pd.concat([train_aug_clean, test_aug_clean], ignore_index=True))
train_aug_fe = fe2.transform(train_aug_clean)
test_aug_fe  = fe2.transform(test_aug_clean)

for col in ['Item_Identifier','Outlet_Identifier','Item_Type','Item_Category','Item_Cat_x_Outlet']:
    if col in train_aug_clean.columns: train_aug_fe[col] = train_aug_clean[col].values
    if col in test_aug_clean.columns:  test_aug_fe[col]  = test_aug_clean[col].values

te2 = KFoldTargetEncoder(n_folds=N_FOLDS, alpha=10.0, seed=SEED)
te2.fit(train_aug_fe, pd.Series(y_aug_log))
train_aug_final = te2.transform_train(train_aug_fe, pd.Series(y_aug_log))
test_aug_final  = te2.transform_test(test_aug_fe)
for c in DROP_FEATURES:
    train_aug_final = train_aug_final.drop(columns=[c], errors='ignore')
    test_aug_final  = test_aug_final.drop(columns=[c], errors='ignore')

print(f'Augmented train size: {len(train_aug_final):,} (was {len(train_final):,})')


In [ ]:
grocery_mask_aug   = (train_aug_clean['Outlet_Type'] == 'Grocery Store').values
grocery_mask_test2 = (test_aug_clean['Outlet_Type']  == 'Grocery Store').values

X_sup2  = train_aug_final[FEATURES][~grocery_mask_aug].values
y_sup2  = y_aug_log[~grocery_mask_aug]
X_groc2 = train_aug_final[FEATURES][grocery_mask_aug].values
y_groc2 = y_aug_log[grocery_mask_aug]

X_test_sup2  = test_aug_final[FEATURES][~grocery_mask_test2].values
X_test_groc2 = test_aug_final[FEATURES][grocery_mask_test2].values

def quick_gbm_blend(params_xgb, params_lgb, params_cat, X_tr, y_tr, X_te,
                    n_folds=5, seed=42):
    """
    3-model blend. Returns (raw-space test predictions, raw-space OOF RMSE).
    Both y_tr and returned RMSE are in raw sales space — no unit mismatch.
    """
    kf = KFold(n_folds, shuffle=True, random_state=seed)
    oof_log = np.zeros(len(y_tr))
    test_preds_log = np.zeros(len(X_te))

    for mtype, params in [('xgb', params_xgb), ('lgb', params_lgb), ('cat', params_cat)]:
        if mtype == 'xgb':   mk = lambda p: xgb.XGBRegressor(**p)
        elif mtype == 'lgb': mk = lambda p: lgb.LGBMRegressor(**p)
        else:                mk = lambda p: cbt.CatBoostRegressor(**p)
        fold_test = []
        for tr, va in kf.split(X_tr):
            m = mk(params); m.fit(X_tr[tr], y_tr[tr])
            oof_log[va] += m.predict(X_tr[va]) / 3
            fold_test.append(m.predict(X_te))
        test_preds_log += np.mean(fold_test, axis=0) / 3

    # Convert OOF and test predictions back to raw sales space
    raw_rmse = np.sqrt(mean_squared_error(np.expm1(y_tr), np.expm1(oof_log)))
    return np.expm1(test_preds_log), raw_rmse   # raw test preds, raw RMSE

preds_sup2,  rmse2_sup  = quick_gbm_blend(
    params_xgb_sup,  params_lgb_sup,  params_cat_sup,  X_sup2,  y_sup2,  X_test_sup2)
preds_groc2, rmse2_groc = quick_gbm_blend(
    params_xgb_groc, params_lgb_groc, params_cat_groc, X_groc2, y_groc2, X_test_groc2)

final_preds2            = np.zeros(len(test_aug_final))
sup_idx2                = np.where(~grocery_mask_test2)[0]
groc_idx2               = np.where(grocery_mask_test2)[0]
final_preds2[sup_idx2]  = preds_sup2
final_preds2[groc_idx2] = preds_groc2
final_preds2            = np.clip(final_preds2, MIN_SALES, None)

# Both rmse2_* and combined_rmse are now in raw sales space — safe to compare
n_aug          = len(y_aug_log)
combined_rmse2 = ((rmse2_sup  * (~grocery_mask_aug).sum() +
                   rmse2_groc *   grocery_mask_aug.sum()) / n_aug)
print(f'Pseudo-label OOF RMSE (raw sales): {combined_rmse2:,.2f}')
print(f'Round-1      OOF RMSE (raw sales): {combined_rmse:,.2f}')

adopt_pseudo = combined_rmse2 < combined_rmse
if adopt_pseudo:
    sub2           = pd.read_csv(SUB_PATH)
    sub2[TARGET]   = final_preds2
    sub2.to_csv(OUT_PATH2, index=False)
    print(f'Pseudo-label ADOPTED → {OUT_PATH2}')
else:
    import shutil
    shutil.copy(OUT_PATH, OUT_PATH2)
    print('Pseudo-label did NOT improve — keeping round-1 predictions.')


## 17 · Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(final_preds, bins=60, color='teal', edgecolor='white')
axes[0].set_title('All Test Predictions (v9)')
axes[0].set_xlabel('Predicted Sales')

axes[1].hist(preds_sup,  bins=50, alpha=0.6, color='steelblue', label='Supermarket')
axes[1].hist(preds_groc, bins=30, alpha=0.6, color='salmon',    label='Grocery')
axes[1].set_title('Predictions by Split')
axes[1].set_xlabel('Predicted Sales')
axes[1].legend()

axes[2].hist(np.log1p(final_preds), bins=60, color='coral', edgecolor='white')
axes[2].set_title('log1p Predictions')
axes[2].set_xlabel('log1p(Predicted Sales)')

plt.tight_layout()
plt.show()
print(f'Sup  mean={preds_sup.mean():.0f}  std={preds_sup.std():.0f}')
print(f'Groc mean={preds_groc.mean():.0f}  std={preds_groc.std():.0f}')


In [ ]:
# Embedding PCA — visualise item embedding space
from sklearn.decomposition import PCA

item_emb   = nn_trainer_sup.get_embeddings(train_clean_sup, 'Item_Identifier')
item_emb_v = item_emb[1:]    # skip padding index 0

pca    = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(item_emb_v)

item_ids  = cat_encoders_nn['Item_Identifier'].classes_
prefixes  = [x[:2] for x in item_ids]
color_map = {'FD': 'steelblue', 'DR': 'orange', 'NC': 'salmon'}
colors    = [color_map.get(p, 'gray') for p in prefixes]

plt.figure(figsize=(9, 6))
for cat, col in color_map.items():
    mask = [p == cat for p in prefixes]
    plt.scatter(coords[mask, 0], coords[mask, 1],
                c=col, alpha=0.5, s=12, label=cat)
plt.title('Item embedding space — PCA(2)  [Supermarket NN]')
plt.xlabel('PC 1');  plt.ylabel('PC 2')
plt.legend()
plt.tight_layout()
plt.show()
print('Items that cluster together sell similarly across all outlet/MRP conditions.')


## 18 · Results Summary

In [ ]:
print('=' * 65)
print('BIGMART v9 — RESULTS SUMMARY')
print('=' * 65)
print(f'  Base GBM features          : {len(FEATURES)}')
print(f'  Embedding features added   : {n_emb_added}')
print(f'  Total GBM features         : {len(FEATURES_EMB)}')
print(f'  Continuous NN features     : {len(CONT_COLS_NN)}')
print()
print('  NN standalone OOF RMSE (log-space, for reference):')
print(f'    Supermarket              : {rmse_nn_sup:.5f}')
print(f'    Grocery                  : {rmse_nn_groc:.5f}')
print()
print('  4-model stacking OOF RMSE (raw sales, honest held-out meta eval):')
print(f'    Supermarket              : {rmse_sup:,.2f}  (n={n_sup:,})')
print(f'    Grocery                  : {rmse_groc:,.2f}  (n={n_groc:,})')
print(f'    Combined                 : {combined_rmse:,.2f}')
print()
print(f'  Pseudo-label OOF RMSE      : {combined_rmse2:,.2f}  (raw, GBM blend)')
print(f'  Pseudo-label adopted       : {adopt_pseudo}')
print()
print(f'  v8 baseline RMSE           : 1174.49')
print(f'  v9 delta (vs v8)           : {combined_rmse - 1174.49:+.2f}')
print()
print('  New in v9 vs v8:')
print('    + EntityEmbeddingNN — 4th base learner')
print('    + Learned embeddings injected into GBM feature set (+34 features)')
print('    + MLP meta-learner (6→32→32→1) replacing BayesianRidge')
print('    + 4-model OOF stacking (XGB / LGB / CAT / NN)')
print('    + Meta RMSE evaluated on held-out fold (not in-sample)')
print('    + quick_gbm_blend RMSE in raw space (apples-to-apples pseudo comparison)')
print()
print(f'  Submission (round-1)       : {OUT_PATH}')
print(f'  Submission (pseudo-label)  : {OUT_PATH2}')
print('=' * 65)
